# U.S. AI Infrastructure — Market Opportunity Analysis

Structured scoring framework for ranking U.S. power markets as development sites for AI energy infrastructure.

| | |
|---|---|
| **Markets** | 8 U.S. ISO/RTO regions |
| **Framework** | 4 pillars · 13 metrics · composite 0–100 opportunity score |
| **Layers** | Baseline scoring · Pillar decomposition · Customer archetypes · Scenario robustness · Risk assessment |
| **Output convention** | Scores range 0–100. Higher = greater development opportunity. |

---

In [ ]:
import sys
from pathlib import Path

# Resolve project root regardless of where the notebook kernel is launched from
ROOT = Path.cwd()
while not (ROOT / 'market_intelligence').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

pd.set_option('display.float_format', '{:.1f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 130)

FIGURES_DIR = ROOT / 'market_intelligence' / 'outputs' / 'figures'
TABLES_DIR  = ROOT / 'market_intelligence' / 'outputs' / 'tables'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from market_intelligence.src.processing import load_markets, normalize_all
from market_intelligence.src.scoring import score_markets
from market_intelligence.src.config import METRICS, PILLAR_WEIGHTS

INPUT_PATH = ROOT / 'market_intelligence' / 'data' / 'raw' / 'market_inputs.csv'

raw_df        = load_markets(INPUT_PATH)
metric_scores = normalize_all(raw_df)
scored_df     = score_markets(raw_df, metric_scores)

scored_df.to_csv(TABLES_DIR / 'market_scores.csv', index=False)

print(f'Pipeline complete  |  {len(scored_df)} markets  |  {len(METRICS)} metrics  |  4 pillars')

In [ ]:
# Baseline scores overview — all markets, all pillars
pillar_cols  = [f'{p}_score' for p in PILLAR_WEIGHTS]
display_cols = ['rank', 'market_name', 'iso_rto', 'opportunity_score'] + pillar_cols

(
    scored_df[display_cols]
    .sort_values('rank')
    .set_index('rank')
    .rename(columns={
        'market_name':       'Market',
        'iso_rto':           'ISO/RTO',
        'opportunity_score': 'Composite',
        'demand_score':      'Demand',
        'energy_score':      'Energy',
        'feasibility_score': 'Feasibility',
        'strategic_score':   'Strategic',
    })
    .style
    .background_gradient(subset=['Composite'], cmap='Blues', vmin=30, vmax=80)
    .format('{:.1f}', subset=['Composite', 'Demand', 'Energy', 'Feasibility', 'Strategic'])
    .set_caption('Baseline Opportunity Scores — All Markets')
    .set_table_styles([{
        'selector': 'caption',
        'props': [('font-weight', 'bold'), ('font-size', '13px'),
                  ('text-align', 'left'), ('padding-bottom', '8px')]
    }])
)

---
## 1. Market Rankings

Composite opportunity score decomposed into weighted pillar contributions. Each bar's total length equals the composite score, so rank order and the mix of driving factors are visible in a single view.

Pillar weights: **Demand 30%** · **Energy 30%** · **Feasibility 25%** · **Strategic Fit 15%**

In [ ]:
from market_intelligence.src.visualization import plot_opportunity_ranking

fig1 = plot_opportunity_ranking(scored_df)
fig1.savefig(FIGURES_DIR / 'opportunity_ranking.png')
plt.show()

---
## 2. Pillar Strength Profile

The heatmap identifies where each market excels or underperforms by pillar — useful for distinguishing broadly attractive markets from single-pillar plays. The radar chart profiles the top four markets for direct comparison across all dimensions.

In [ ]:
from market_intelligence.src.visualization import plot_pillar_heatmap

fig2 = plot_pillar_heatmap(scored_df)
fig2.savefig(FIGURES_DIR / 'pillar_heatmap.png')
plt.show()

In [ ]:
from market_intelligence.src.visualization import plot_pillar_radar

fig3 = plot_pillar_radar(
    scored_df,
    # To compare specific markets, pass market_ids explicitly:
    # market_ids=['ercot_tx', 'pnw_wa', 'miso_ia', 'caiso_ca'],
)
fig3.savefig(FIGURES_DIR / 'pillar_radar.png')
plt.show()

---
## 3. Customer Archetype Analysis

Market rankings shift when pillar weights reflect different tenant cost structures. Three archetypes are modeled:

| Archetype | Demand | Energy | Feasibility | Strategic | Primary driver |
|---|---|---|---|---|---|
| Hyperscaler | 40% | 25% | 25% | 10% | Execution at scale, ecosystem depth |
| AI Lab | 15% | 50% | 25% | 10% | Power cost (dominant operating expense) |
| Enterprise | 45% | 20% | 25% | 10% | Demand proximity, latency |

Markets with long bars on all three archetypes are broadly attractive. Markets with a long bar on only one archetype are tenant-specific opportunities.

In [ ]:
from market_intelligence.src.visualization_extended import plot_archetype_rankings

fig4 = plot_archetype_rankings(metric_scores, raw_df)
fig4.savefig(FIGURES_DIR / 'archetype_rankings.png')
plt.show()

---
## 4. Scenario Robustness

Tests whether rankings hold under three alternative structural conditions:

- **High AI Demand Growth** — demand pillar up-weighted; capacity inputs scaled +40%
- **Rising Power Prices (+35%)** — energy pillar up-weighted; all prices scaled +35%
- **Tighter Grid Constraints** — feasibility up-weighted; interconnection queues extended +50%

Flat lines = robust markets whose ranking is stable regardless of which scenario materializes. Steep lines = markets with a scenario-dependent thesis.

In [ ]:
from market_intelligence.src.visualization_extended import plot_scenario_robustness
from market_intelligence.src.scenarios import most_robust_markets

fig5 = plot_scenario_robustness(raw_df)
fig5.savefig(FIGURES_DIR / 'scenario_robustness.png')
plt.show()

print('Most robust markets (top half ranking across all scenarios):')
robust = most_robust_markets(raw_df)
print(robust[['market_name', 'avg_rank', 'worst_rank', 'rank_range']].to_string(index=False))

---
## 5. Risk Assessment

Opportunity and risk are assessed independently — a high-opportunity market can carry material execution risk. Three dimensions are rated per market: **Regulatory**, **Grid**, and **Execution**.

The 2×2 matrix positions markets by opportunity score (x-axis) and aggregate risk (y-axis). The **bottom-right quadrant** (High Opportunity · Low Risk) represents the clearest capital allocation case. The **top-right quadrant** (High Opportunity · High Risk) requires a specific capability advantage to be actionable.

In [ ]:
from market_intelligence.src.visualization_extended import plot_opportunity_risk_matrix
from market_intelligence.src.risk_assessment import risk_matrix

fig6 = plot_opportunity_risk_matrix(scored_df)
fig6.savefig(FIGURES_DIR / 'opportunity_risk_matrix.png')
plt.show()

In [ ]:
# Risk matrix detail — one row per market, three risk dimensions
risk_df = risk_matrix()

_RISK_PALETTE = {'Low': 'background-color: #d4edda', 'Medium': 'background-color: #fff3cd', 'High': 'background-color: #f8d7da'}

(
    risk_df
    .set_index('market_id')
    .rename(columns={
        'regulatory_risk': 'Regulatory',
        'grid_risk':       'Grid',
        'execution_risk':  'Execution',
        'aggregate_risk':  'Aggregate',
    })
    .style
    .applymap(lambda v: _RISK_PALETTE.get(v, ''), subset=['Regulatory', 'Grid', 'Execution', 'Aggregate'])
    .set_caption('Risk Assessment by Market and Dimension')
    .set_table_styles([{
        'selector': 'caption',
        'props': [('font-weight', 'bold'), ('font-size', '13px'),
                  ('text-align', 'left'), ('padding-bottom', '8px')]
    }])
)

---
## 6. Final Recommendations

Markets are classified into four action categories based on the opportunity/risk 2×2 framework:

| Category | Criterion | Interpretation |
|---|---|---|
| **Priority** | High opportunity · Low risk | Clearest capital allocation case; pursue actively |
| **Proceed with Caution** | High opportunity · High risk | Actionable only with a specific capability advantage |
| **Monitor** | Low opportunity · Low risk | Revisit if market conditions change |
| **Avoid** | Low opportunity · High risk | Both dimensions unfavorable under current conditions |

In [ ]:
from market_intelligence.src.visualization import export_full_results

full_results = export_full_results(
    scored_df,
    output_path=TABLES_DIR / 'full_results.csv',
)

_CATEGORY_PALETTE = {
    'Priority':              'background-color: #d4edda; font-weight: bold',
    'Proceed with Caution':  'background-color: #fff3cd',
    'Monitor':               'background-color: #f8f9fa; color: #666',
    'Avoid':                 'background-color: #f8d7da; color: #666',
}

score_cols = [c for c in ['Opportunity Score', 'Demand Score', 'Energy Score',
                           'Feasibility Score', 'Strategic Score'] if c in full_results.columns]
display_cols = ['market_name'] + score_cols + ['aggregate_risk', 'recommendation']
display_cols = [c for c in display_cols if c in full_results.columns]

(
    full_results.set_index('rank')[display_cols]
    .rename(columns={'market_name': 'Market', 'aggregate_risk': 'Risk', 'recommendation': 'Category'})
    .style
    .background_gradient(subset=['Opportunity Score'], cmap='Blues', vmin=30, vmax=80)
    .applymap(lambda v: _CATEGORY_PALETTE.get(v, ''), subset=['Category'])
    .format('{:.1f}', subset=score_cols)
    .set_caption(f'Final Market Recommendations  |  Exported to: full_results.csv')
    .set_table_styles([{
        'selector': 'caption',
        'props': [('font-weight', 'bold'), ('font-size', '13px'),
                  ('text-align', 'left'), ('padding-bottom', '8px')]
    }])
)

---
## Export Summary

All outputs are written to organized directories under `market_intelligence/outputs/`. Run the cell below to confirm paths and file sizes.

In [ ]:
# Confirm all saved outputs
all_outputs = sorted(list(FIGURES_DIR.glob('*.png')) + list(TABLES_DIR.glob('*.csv')))

print(f'Figures:  {FIGURES_DIR}')
print(f'Tables:   {TABLES_DIR}\n')

for path in all_outputs:
    size_kb = path.stat().st_size / 1024
    folder  = 'figures' if path.suffix == '.png' else 'tables '
    print(f'  [{folder}]  {path.name:<40s}  {size_kb:6.1f} KB')

---
## Appendix: Weight Sensitivity

The core ranking is stable to moderate weight changes, but individual markets shift meaningfully. The cell below re-scores under an energy-developer weighting scheme (energy = 45%) to illustrate directional sensitivity.

A positive `rank_delta` means the market improves under energy-weighted scoring — i.e., its power cost advantage becomes more decisive.

In [ ]:
import market_intelligence.src.config as cfg

_original = cfg.PILLAR_WEIGHTS.copy()
cfg.PILLAR_WEIGHTS = {'demand': 0.15, 'energy': 0.45, 'feasibility': 0.25, 'strategic': 0.15}
scored_energy = score_markets(raw_df, metric_scores)
cfg.PILLAR_WEIGHTS = _original

sensitivity = (
    scored_df[['market_id', 'market_name', 'rank', 'opportunity_score']]
    .rename(columns={'rank': 'rank_baseline', 'opportunity_score': 'score_baseline'})
    .merge(
        scored_energy[['market_id', 'rank', 'opportunity_score']]
        .rename(columns={'rank': 'rank_energy45', 'opportunity_score': 'score_energy45'}),
        on='market_id',
    )
    .assign(rank_delta=lambda d: d['rank_baseline'] - d['rank_energy45'])
    .sort_values('rank_baseline')
    .set_index('rank_baseline')
    .drop(columns=['market_id'])
)

(
    sensitivity
    .rename(columns={
        'market_name':    'Market',
        'score_baseline': 'Score (baseline)',
        'rank_energy45':  'Rank (energy 45%)',
        'score_energy45': 'Score (energy 45%)',
        'rank_delta':     '\u0394 Rank',
    })
    .style
    .bar(subset=['\u0394 Rank'], align='zero', color=['#c0392b', '#1e6348'])
    .format('{:.1f}', subset=['Score (baseline)', 'Score (energy 45%)'])
    .set_caption(
        'Baseline (demand=30%, energy=30%)  vs.  Energy-Weighted (energy=45%)  '
        '|  \u0394 Rank > 0 = improves under energy weighting'
    )
    .set_table_styles([{
        'selector': 'caption',
        'props': [('font-weight', 'bold'), ('font-size', '12px'),
                  ('text-align', 'left'), ('padding-bottom', '6px')]
    }])
)